In [97]:
import numpy as np
import pandas as pd
import math
import optuna
from tqdm import tqdm
import gc
import psutil
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn import TransformerDecoder, TransformerDecoderLayer, TransformerEncoder, TransformerEncoderLayer
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import matthews_corrcoef
from transformers import AutoTokenizer, AutoModel
from functools import lru_cache
from torch.amp import GradScaler, autocast
import pickle

#Clear the GPU memory cache
torch.cuda.empty_cache()
pd.options.mode.chained_assignment = None  # Suppress the warning globally

import wandb
import os
import json

from torchcrf import CRF
import ast
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import numpy as np
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import logging
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)
logging.getLogger("torch._inductor").setLevel(logging.ERROR)


#Clear the GPU memory cache
torch.cuda.empty_cache()

#Configure CUDA memory allocations (helps manage fragmentation in the GPU memory)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [98]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device_type = device.type  # "cuda", "mps", or "cpu"

if device_type == "mps":
    #input_data_dir_path = "../../../data/processed_data/model_data/shared_crf/model_with_errors"
    num_workers_cpu = 0
    pin_memory = False
elif device_type == "cuda":
    #input_data_dir_path = "/tmp/nrt204/FragmentPredictor3/data/processed_data/model_data/shared_crf/model_with_errors" #n SCARB Cluster
    num_workers_cpu = 4
    pin_memory = True

print("Device type:", device_type)

#Insert choise here
model_choice = "esm2_8M_nt"

##Integrate different model choice options?
if model_choice == "esm2_8M_nt":
    #8M ESM-2 with codon encoding
    esm2_model_name = "facebook/esm2_t6_8M_UR50D"
    model_name_ckpt = "full_model_200_genomes_trained.pth" #"esm2_t6_8M_nt.pth" #MODIFY


esm2_model_abbr = esm2_model_name.split("/")[-1].split("_UR")[0]
esm2_model = AutoModel.from_pretrained(esm2_model_name)

max_aa_len = 100
max_len = max_aa_len + 2 #Add CLS and EOS tokens

test_samples_file = open("../../../data/processed_data/genome_partitions/test_partition_accessions.txt", "r")
test_samples = [line.strip() for line in test_samples_file.readlines()]
test_samples = test_accessions = ['GCF_001277215.2', 'GCF_000006765.1', "GCF_000012765.1"] #REMOVE
test_samples_file.close()

Device type: mps


/Users/nrt204/miniconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [99]:
def clear_memory():
    #Memory clean up function
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

In [100]:
genetic_code = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': 'X', 'TAG': 'X',
    'TGT': 'C', 'TGC': 'C', 'TGA': 'X', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_nucleotide_to_amino_acid(sequence):
    """
    The function takes a nucleotide sequence and translates it into the corresponding amino acid sequence. 
    
    Args:
        sequence (str): A nucleotide sequence.
        
    Returns:
        (str): A string representing the amino acid sequence translated from the 
               nucleotide input. Each codon is mapped to its corresponding amino acid,
               with stop codons represented by "X".
    """

    sequence = str(sequence)
    
    # More efficient length adjustment
    remainder = len(sequence) % 3
    if remainder:
        sequence = sequence[:-remainder]
    
    # Direct list comprehension is fastest
    return ''.join(genetic_code.get(sequence[i:i+3], "<mask>") 
                   for i in range(0, len(sequence), 3))

In [101]:
def one_hot_encode(sequence):
    """
    One-hot encode nucleotide sequences in a matrix format of 4 rows (A, C, G, T)
    and len(sequence) columns.

    Args:
        sequence (str): a nucleotide sequence. 

    Returns: 
        encoding (tensor): one-hot encoded sequence. 
    """

    #Define the mapping of nucleotides to indices
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    #Create an empty one-hot encoding tensor
    encoding = torch.zeros(4, len(sequence))
    
    #Convert the sequence to a tensor of indices for efficient indexing
    indices = torch.tensor(
        [mapping[char] for char in sequence if char in mapping], dtype=torch.long)
    
    #Use advanced indexing to set the appropriate positions to 1
    positions = torch.arange(len(sequence))[[char in mapping for char in sequence]]
    encoding[indices, positions] = 1
    #For 'N', we do nothing, so the corresponding column remains all zeros

    return torch.tensor(encoding)


In [102]:
def process_nt_sequences_to_codons(nt_sequences, max_aa_len):
    """
    Convert nucleotide sequences from dimension (4, nucleotide_seq_len) to dimension (12, nucleotide_seq_len/3) format
    by grouping every 3 nucleotides (1 codon) together.

    Args:
        nt_sequences: List of tensors with shape (4, nucleotide_seq_len)

    Returns:
        List of tensors with shape (12, nucleotide_seq_len/3)
    """
    processed_sequences = []

    for seq in nt_sequences:
        #seq has shape (4, nucleotide_seq_len) -> reshape to (4, nucleotide_seq_len/3, 3) to group every 3 nucleotides
        seq_reshaped = seq.view(4, max_aa_len, 3)

        #Transpose to (nucleotide_seq_len/3, 4, 3) then reshape to (nucleotide_seq_len/3, 12)
        seq_transposed = seq_reshaped.transpose(0, 1)  # (nucleotide_seq_len/3, 4, 3)
        seq_formatted = seq_transposed.reshape(max_aa_len, 12)  # (nucleotide_seq_len/3, 12)

        processed_sequences.append(seq_formatted)

    return processed_sequences

In [103]:
class SeqDataset(torch.utils.data.Dataset):
    """
    Optimized dataset class with reduced memory overhead.

    Args:
        nt_encodings_rf0 (list): List of nucleotide codon encodings (max_aa_len, 12) for reading frame 0
        aa_encodings_rf0 (BatchEncoding): Tokenized amino acid encodings (dict) for reading frame 0
        nt_encodings_rf1 (list): List of nucleotide codon encodings (max_aa_len, 12) for reading frame 1
        aa_encodings_rf1 (BatchEncoding): Tokenized amino acid encodings (dict) for reading frame 1
        nt_encodings_rf2 (list): List of nucleotide codon encodings (max_aa_len, 12) for reading frame 2
        aa_encodings_rf2 (BatchEncoding): Tokenized amino acid encodings (dict) for reading frame 2
        seq_errors: indel errors in sequence
        cds_coords: CDS coordinates on read
        read_name: unique read identifier for genome accession
    """
    def __init__(self, nt_encodings_rf0, aa_encodings_rf0,
                 nt_encodings_rf1, aa_encodings_rf1, 
                 nt_encodings_rf2, aa_encodings_rf2,
                 seq_errors, cds_coords, read_name):
        
        self.nt_encodings_rf0 = nt_encodings_rf0
        self.aa_encodings_rf0 = aa_encodings_rf0

        self.nt_encodings_rf1 = nt_encodings_rf1
        self.aa_encodings_rf1 = aa_encodings_rf1

        self.nt_encodings_rf2 = nt_encodings_rf2
        self.aa_encodings_rf2 = aa_encodings_rf2

        self.seq_errors = seq_errors
        self.cds_coords = cds_coords
        self.read_name = read_name

    
    def __getitem__(self, idx):
        item = {
            'nt_encodings_rf0': torch.as_tensor(self.nt_encodings_rf0[idx], dtype=torch.float32),
            'aa_encodings_rf0': {key: val[idx] for key, val in self.aa_encodings_rf0.items()},

            'nt_encodings_rf1': torch.as_tensor(self.nt_encodings_rf1[idx], dtype=torch.float32),
            'aa_encodings_rf1': {key: val[idx] for key, val in self.aa_encodings_rf1.items()},

            'nt_encodings_rf2': torch.as_tensor(self.nt_encodings_rf2[idx], dtype=torch.float32),
            'aa_encodings_rf2': {key: val[idx] for key, val in self.aa_encodings_rf2.items()},

            'seq_errors': str(self.seq_errors[idx]),
            'cds_coords': self.cds_coords[idx],
            'read_name': self.read_name[idx]
        }
        return item
    
    def __len__(self):
        return len(self.seq_errors)


In [104]:
def encode_data(processed_samples_df, max_len, tokenizer=None, max_aa_len=max_aa_len):
    """ 
    Encode data samples to fit model input format. 

    Args:
        processed_samples_df (dataframe): Dataframe with input dataset.
        max_len (int): Max ESM model input length
        tokenizer (AutoTokenizer): Specific ESM tokenizer
        max_aa_len (int): maximum amino acid input length; max_len without special tokens (CLS and EOS)

    Returns:
        - dataset (dict): nested dictionary with data formatted to fit model input.
        - list of sequence types. 
    """

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(
            "facebook/esm2_t6_8M_UR50D",
            do_lower_case=False,
        )
    
    encodings_nt = {}
    encodings_aa = {}
    max_nt_len = max_aa_len * 3
    
    #Proces data from each RF separately 
    for rf in ["rf0", "rf1", "rf2"]:
        #====Nucleotide sequence processing====#
        # Pad the sequences

        if rf == "rf0":
            processed_samples_df[f"{rf}_seq_nt"] = processed_samples_df[f"read"]
        elif rf == "rf1":
            processed_samples_df[f"{rf}_seq_nt"] = processed_samples_df[f"read"].apply(lambda seq: seq[1:])
        elif rf == "rf2":
            processed_samples_df[f"{rf}_seq_nt"] = processed_samples_df[f"read"].apply(lambda seq: seq[2:])

        #====Amino acid sequence processing====#
        processed_samples_df[f"{rf}_seq_aa"] = processed_samples_df[f"{rf}_seq_nt"].apply(lambda seq: translate_nucleotide_to_amino_acid(seq))
        aa_sequences = processed_samples_df[f"{rf}_seq_aa"].tolist()

        #Tokenize with strict length control
        aa_encodings_rf = tokenizer(
            aa_sequences,
            padding="max_length",      #Pad all to max_length
            max_length=max_len,        #CLS + 100 AA + EOS = 102
            truncation=True,           #Cut longer sequences
            return_tensors="pt",       #Return PyTorch tensors
        )

        encodings_aa[rf] = aa_encodings_rf

        #====Nucleotide sequence processing continued====#
        processed_samples_df[f"{rf}_seq_nt"] = processed_samples_df[f"{rf}_seq_nt"].apply(lambda seq: seq + 'N' * (max_nt_len - len(seq)) if len(seq) < max_nt_len else seq)
        
        nt_sequences = [one_hot_encode(seq) for seq in processed_samples_df[f"{rf}_seq_nt"]] #[max_nt_len, 4, num_seqs]

        # Process nt_sequences to codon-based format (3*4=12, max_nt_len/3)
        nt_encodings_rf = process_nt_sequences_to_codons(nt_sequences, max_aa_len)
        encodings_nt[rf] = nt_encodings_rf

        cds_coords = processed_samples_df["cds_coords"]
        read_name = processed_samples_df["read_name"]
        seq_errors = processed_samples_df["indel_positions"]

    dataset = SeqDataset(encodings_nt["rf0"], encodings_aa["rf0"],
                         encodings_nt["rf1"], encodings_aa["rf1"],
                         encodings_nt["rf2"], encodings_aa["rf2"],
                         seq_errors, cds_coords, read_name)
    
    return dataset

# Load Models

In [105]:
class SequenceEncoder(nn.Module):
    """
    Sequence encoder using the pretrained ESM-2.
    
    Args:
        esm2_model (str): Name or path of the pretrained ESM-2 model to load
        dropout_rate_1 (float): Dropout rate for regularization layer applied after ESM-2
    
    Attributes:
        pretrained_model_aa (AutoModel): Pretrained ESM-2 model with first half of layers frozen
        dropout_1 (nn.Dropout): Dropout layer for regularization before transformer head
    """
    def __init__(self,
                 esm2_model,
                 dropout_rate_1): 
        super(SequenceEncoder, self).__init__()

        #Load pretrained ESM-2 model for amino acid sequences
        self.pretrained_model_aa = AutoModel.from_pretrained(esm2_model)

        #Freeze first half of layers of ESM-2
        for i, layer in enumerate(self.pretrained_model_aa.encoder.layer):
            if i < len(self.pretrained_model_aa.encoder.layer) // 2:
                for param in layer.parameters():
                    param.requires_grad = False

        #Additional dropout layer for regularization after encoding sequences
        self.dropout_1 = nn.Dropout(dropout_rate_1)

    def forward(self, x_aa, attention_mask_aa):
        """
        Forward pass through the sequence encoder.
    
        Args:
            x_aa (torch.Tensor): Tokenized amino acid sequences of shape (batch_size, seq_len)
            attention_mask_aa (torch.Tensor): Attention mask of shape (batch_size, seq_len)
            
        Returns:
            embeddings_aa (torch.Tensor): Amino acid embeddings with CLS/EOS removed, shape (batch_size, seq_len-2, hidden_size)
            attention_mask_trimmed (torch.Tensor): Attention mask with CLS/EOS removed, shape (batch_size, seq_len-2)
        """
        #Extract features from pretrained ESM-2 model
        features_aa = self.pretrained_model_aa(x_aa, attention_mask=attention_mask_aa)

        #Get last hidden state: [batch_size, tokens, hidden_size]
        sequence_output_aa = features_aa['last_hidden_state']

        #Remove CLS and EOS token embeddings: [batch_size, aa_seq_len, hidden_size]
        sequence_output_aa = sequence_output_aa[:, 1:-1, :]

        #Apply dropout before transformer head
        embeddings_aa = self.dropout_1(sequence_output_aa)

        #Remove CLS/EOS from attention mask for transformer head
        attention_mask_trimmed = attention_mask_aa[:, 1:-1]

        return embeddings_aa, attention_mask_trimmed

class TransformerEncoderBlock(nn.Module):
    """
    Neural network module that applies a Transformer encoder block to encoded sequences and adds a linear layer to fit CRF input.

    Args:
        hidden_size (int): The dimensionality of the input and output features for the Transformer encoder.
        num_layers (int): Number of Transformer encoder layers to stack.
        n_attention_heads (int): Number of attention heads in each Transformer encoder layer.
        dropout_rate_encoder (float): Dropout rate applied after normalization and within the encoder layers.
        act_function (str or Callable): Activation function to use in the feedforward network of the encoder layers.
        num_labels (int): Number of output classes 

    Attributes:
        encoder (nn.TransformerEncoder): Stacked Transformer encoder layers.
        classifier (nn.Linear): Linear layer mapping the encoder output to class logits.
        norm (nn.LayerNorm): Layer normalization applied after the encoder.
        dropout (nn.Dropout): Dropout layer applied after normalization.
        layers (int): Number of encoder layers.
    """
    def __init__(self,
                 hidden_size,
                 num_layers,
                 n_attention_heads,
                 dropout_rate_encoder,
                 act_function,
                 num_labels):
        super().__init__()

        hidden_size_merged = hidden_size + 12 #12 for codon one-hot encoded; hidden_size for amino acid representation from ESM2

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size_merged, 
            nhead=n_attention_heads,
            dim_feedforward=4*hidden_size_merged,
            dropout=dropout_rate_encoder,
            activation=act_function
        )

        self.layers = num_layers
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.linear = nn.Linear(hidden_size_merged, num_labels)
        self.norm = nn.LayerNorm(hidden_size_merged)


    def forward(self, encoded_seqs_nt, encoded_embeddings_aa, trimmed_attention_mask):
        """
        Forward pass through the Transformer encoder block and RF-specific linear classifier.

        Args:
            encoded_seqs_nt (torch.Tensor): Nucleotide encodings with shape (batch_size, seq_len, 12) (one-hot encoded codons).
            encoded_embeddings_aa (torch.Tensor): Amino acid embeddings with shape (batch_size, seq_len, hidden_size) from pretrained ESM-2
            trimmed_attention_mask (torch.Tensor): Boolean mask of shape (batch_size, seq_len) where 1s are valid tokens and 0s are padding positions.

        Returns:
            torch.Tensor: 
                Logits of shape (batch_size, seq_len, num_labels) representing class scores for each token position.
        """

        #Concatenate ESM-2 embeddings and one hot encoded codons
        combined_codon_and_aa_embeddings = torch.cat([encoded_embeddings_aa, encoded_seqs_nt], dim=-1)

        combined_codon_and_aa_embeddings = combined_codon_and_aa_embeddings.permute(1, 0, 2)  # [seq_len, batch, hidden + 12]
        attention_mask_transformer = ~trimmed_attention_mask.bool()

        #Pass through transformer encoder layers
        combined_codon_and_aa_embeddings = self.encoder(combined_codon_and_aa_embeddings, src_key_padding_mask=attention_mask_transformer)

        combined_codon_and_aa_embeddings = combined_codon_and_aa_embeddings.permute(1, 0, 2)  # [batch, seq_len, hidden + 12]

        #Apply layer normalization
        combined_codon_and_aa_embeddings = self.norm(combined_codon_and_aa_embeddings)

        #Get RF-specific class logits out
        logits = self.linear(combined_codon_and_aa_embeddings)  # [batch, seq_len, num_labels]

        return logits


class LinearChainCRF(nn.Module):
    """
    Neural network module that applies a Conditional Random Field (CRF) layer for structured prediction.
    Designed to handle dynamic reading-frame (RF) combination labels and enforce biologically 
    constrained transitions between RF states.

    Args:
        mapping_dict_to_class (dict): Mapping from integer label indices to tuples representing the corresponding reading-frame combination (e.g., `{0: (0, 1, 2), 1: (1, 3, 5), ...}`).
        class_weights (list[float] or torch.Tensor or None): Class-specific weights for balancing the loss. If None, all classes are equally weighted.
        num_class_labels (int, optional): Total number of class labels. If not provided, it is inferred from `mapping_dict_to_class`.

    Attributes:
        shared_rf_labels_mapping (dict): Stores the mapping from label indices to RF combinations.
        class_weights (torch.Tensor): Buffer tensor of class weights used for weighted CRF loss computation.
        crf (torchcrf.CRF): Linear-chain Conditional Random Field layer that models label dependencies across the sequence.
        legal_transitions (dict): Defines valid transitions between RF states for each of the three reading frames.
        biologically_valid_mask (torch.BoolTensor): Mask indicating which transitions are allowed (True) or constrained (False).
        frequent_transition_mask (torch.BoolTensor): Mask indicating which transitions are frequent self-transitions.
    """

    def __init__(self, 
                 mapping_dict_to_class, 
                 num_class_labels=None):
        super().__init__()

        #Load shared class mapping
        self.shared_rf_labels_mapping = mapping_dict_to_class

        #Determine number of classes if not provided
        if num_class_labels is None:
            num_class_labels = len(self.shared_rf_labels_mapping)

        self.crf = CRF(num_tags=num_class_labels, batch_first=True)

        #Define allowed RF transition rules
        self.legal_transitions = {
            0: {0, 2, 4},
            1: {1, 3, 5},
            2: {1, 5},
            3: {0, 2, 4},
            4: {1, 3, 5},
            5: {0, 2, 4}
        }

        self.biologically_valid_mask = torch.ones_like(self.crf.transitions, dtype=torch.bool)
        self.frequent_transition_mask = torch.zeros_like(self.crf.transitions, dtype=torch.bool)
        
        self._create_biologically_valid_mask()
        self._create_frequent_transition_mask()

        #Initialize transitions with three-tier scheme
        #with torch.no_grad():
            # Stage 1: All illegal transitions → -10 (forbidden)
        #    self.crf.transitions[~self.biologically_valid_mask] = -10
            
            # Stage 2: Legal but infrequent transitions → -2 (discouraged)
        #    self.crf.transitions[self.biologically_valid_mask] = -2
            
            # Stage 3: Frequent self-transitions → 0 (neutral/encouraged)
        #    self.crf.transitions[self.frequent_transition_mask] = 0 

    def _is_legal_transition(self, from_rf, to_rf):
        """Check if transition from one RF combination to another is legal"""
        from_rf0, from_rf1, from_rf2 = from_rf
        to_rf0, to_rf1, to_rf2 = to_rf

        # All three RFs must have legal transitions
        rf0_legal = to_rf0 in self.legal_transitions[from_rf0]
        rf1_legal = to_rf1 in self.legal_transitions[from_rf1]
        rf2_legal = to_rf2 in self.legal_transitions[from_rf2]

        return rf0_legal and rf1_legal and rf2_legal

    def _create_biologically_valid_mask(self):
        """
        Create a mask indicating which transitions should be constrained.
        False = constrained (will be set to penalty), True = learnable
        """
        num_labels = len(self.shared_rf_labels_mapping)
        legal_count = 0
        illegal_count = 0

        # Check all possible transitions between the labels
        for from_label in range(num_labels):
            for to_label in range(num_labels):
                from_rf = self.shared_rf_labels_mapping[from_label]
                to_rf = self.shared_rf_labels_mapping[to_label]

                if self._is_legal_transition(from_rf, to_rf):
                    # Legal transition - keep as learnable (True)
                    self.biologically_valid_mask[from_label, to_label] = True
                    legal_count += 1
                else:
                    # Illegal transition - mark as constrained (False)
                    self.biologically_valid_mask[from_label, to_label] = False
                    illegal_count += 1

        print(f"Legal transitions: {legal_count}")
        print(f"Illegal transitions: {illegal_count}")
        print(f"Total transitions: {legal_count + illegal_count}")
        print(f"Percentage legal: {legal_count/(legal_count + illegal_count)*100:.1f}%")

    def _create_frequent_transition_mask(self):
        """
        Create a mask for frequent self-transitions.
        These are transitions from a label to itself for the special RF combinations:
        (0,0,0)->(0,0,0), (1,0,0)->(1,0,0), (0,1,0)->(0,1,0), (0,0,1)->(0,0,1)
        """
        # Define the frequent RF combinations
        frequent_rf_combinations = {(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)}
        
        # Find label indices corresponding to these RF combinations
        frequent_labels = []
        for label_idx, rf_combo in self.shared_rf_labels_mapping.items():
            if rf_combo in frequent_rf_combinations:
                frequent_labels.append(label_idx)
        
        # Mark self-transitions for these labels
        frequent_count = 0
        for label in frequent_labels:
            self.frequent_transition_mask[label, label] = True
            frequent_count += 1
        
        print(f"Frequent self-transitions: {frequent_count}")
        print(f"Frequent transition labels: {frequent_labels}")

    def forward(self, logits, attention_mask, labels=None):
        """
        Forward pass with CRF layer.

        Args: 
            logits (torch.Tensor): Emission scores of shape (batch_size, seq_len, num_labels).
            attention_mask (torch.Tensor): Mask where 1 indicates valid tokens.
            labels (torch.Tensor, optional): Gold label indices for training.
            sequence_class (torch.Tensor, optional): Sequence-level class indices for weighted loss.

        Returns: 
            dict: 
            During training (`labels` provided):
                    - **'loss' (torch.Tensor)**: Mean CRF loss over the batch.
                    - **'logits' (torch.Tensor)**: Input logits passed through the CRF.
                During inference (`labels` omitted):
                    - **'predictions' (list[list[int]])**: Decoded most probable label sequence per sample.
                    - **'logits' (torch.Tensor)**: Input logits passed through the CRF.
        """
        #Training
        if labels is not None:
            crf_mask = attention_mask.bool()
            #calculate log likelihood of the true label sequence under the CRF model for each sequence in batch (no reduction across batch)
            log_likelihood = self.crf(logits, labels, mask=crf_mask, reduction='none') #returns the log-likelihood value per true label sequence 

            #compute negative log likelihood loss averaged across batch, optionally weighted per sequence
            loss = -log_likelihood.mean()

            return {'loss': loss, 'logits': logits}

        #Inference mode: decode the most probable label sequence using the Viterbi algorithm
        else:
            crf_mask = attention_mask.bool()
            predictions = self.crf.decode(logits, mask=crf_mask)
            return {'predictions': predictions, 'logits': logits}


class CDSPredictor(nn.Module):
    """
    Full model for CDS prediction combining:
      - Pretrained ESM-2 amino acid embeddings,
      - Transformer encoders per reading frame (RF0–RF2),
      - And a CRF layer for structured, frame-consistent predictions.

    Args:
        esm2_model (nn.Module): Pretrained ESM-2 model used to extract amino acid embeddings.
        num_layers (int): Number of Transformer encoder layers per reading frame.
        n_attention_heads (int): Number of attention heads in each Transformer layer.
        dropout_rate_1 (float): Dropout rate applied in the sequence encoder.
        dropout_rate_2 (float): Dropout rate applied within the Transformer encoder layers.
        act_function (str or Callable): Activation function used in Transformer feedforward layers.
        class_weights (list[float] or torch.Tensor): Optional weights to balance CRF loss across sequence classes.
        num_encoded_labels (int): Number of combined label states used by the CRF.
        encoded_labels_mapping (dict): Mapping from integer label indices to RF combination tuples.

    Attributes:
        sequence_encoder (SequenceEncoder): Module that extracts amino acid embeddings from the pretrained ESM-2 model.
        TransformerEncoderBlock (TransformerEncoderBlock): Transformer encoder applied independently to each reading frame.
        linear_transform (nn.Linear): Linear projection layer mapping concatenated RF outputs (3*num_labels) to num_encoded_labels.
        CRF (LinearChainCRF): Conditional Random Field layer enforcing structured transitions between predicted RF combinations.
    """

    def __init__(self,
                 esm2_model,
                 num_layers,
                 n_attention_heads,
                 dropout_rate_1,
                 dropout_rate_2,
                 act_function,
                 num_encoded_labels,
                 encoded_labels_mapping
                 ): 
        super(CDSPredictor, self).__init__()

        #Extract amino acid representations from pretrained ESM-2 model
        self.sequence_encoder = SequenceEncoder(
            esm2_model,
            dropout_rate_1)

        #Transformer encoder block applied separately to each reading frame
        self.TransformerEncoderBlock = TransformerEncoderBlock(
            hidden_size=self.sequence_encoder.pretrained_model_aa.config.hidden_size,
            num_layers=num_layers,
            n_attention_heads=n_attention_heads,
            dropout_rate_encoder=dropout_rate_2,
            act_function=act_function,
            num_labels=6)

        #Linear layer to combine outputs from the 3 reading frames (3 * 6 logits -> num_encoded_labels)
        self.linear_transform = nn.Linear(3*6, num_encoded_labels)

        #CRF layer for structured prediction with transition constraints
        self.CRF = LinearChainCRF(mapping_dict_to_class = encoded_labels_mapping,
                                  num_class_labels=num_encoded_labels)


    def forward(self, encoded_seqs_nt_rf0, x_aa_rf0, attention_mask_aa_rf0, 
                      encoded_seqs_nt_rf1, x_aa_rf1, attention_mask_aa_rf1, 
                      encoded_seqs_nt_rf2, x_aa_rf2, attention_mask_aa_rf2, 
                      labels=None):
        """
        Forward pass through the full CDS prediction model.

        Args:
            encoded_seqs_nt_rf{0,1,2} (torch.Tensor): One-hot codon encodings for each reading frame, shape (batch_size, seq_len, 12).
            x_aa_rf{0,1,2} (torch.Tensor): Amino acid token embeddings for each RF, shape (batch_size, seq_len, hidden_size).
            attention_mask_aa_rf{0,1,2} (torch.Tensor): Boolean masks marking valid tokens, shape (batch_size, seq_len).
            labels (torch.Tensor, optional): Ground-truth encoded, shared labels for CRF training.
            sequence_class (torch.Tensor, optional): Sequence-level class indices for weighted CRF loss.

        Returns:
            dict:
                If training → {'loss': torch.Tensor, 'logits': torch.Tensor}
                If inference → {'predictions': list[list[int]], 'logits': torch.Tensor}

        """

        #Encode amino acid sequences for each reading frame 
        encoded_embeddings_aa_rf0, trimmed_attention_mask_rf0 = self.sequence_encoder(x_aa_rf0, attention_mask_aa_rf0)
        encoded_embeddings_aa_rf1, trimmed_attention_mask_rf1 = self.sequence_encoder(x_aa_rf1, attention_mask_aa_rf1)
        encoded_embeddings_aa_rf2, trimmed_attention_mask_rf2 = self.sequence_encoder(x_aa_rf2, attention_mask_aa_rf2)

        #Process each RF through its transformer encoder blocks
        logits_rf0 = self.TransformerEncoderBlock(encoded_seqs_nt=encoded_seqs_nt_rf0, encoded_embeddings_aa=encoded_embeddings_aa_rf0, trimmed_attention_mask=trimmed_attention_mask_rf0)
        logits_rf1 = self.TransformerEncoderBlock(encoded_seqs_nt=encoded_seqs_nt_rf1, encoded_embeddings_aa=encoded_embeddings_aa_rf1, trimmed_attention_mask=trimmed_attention_mask_rf1)
        logits_rf2 = self.TransformerEncoderBlock(encoded_seqs_nt=encoded_seqs_nt_rf2, encoded_embeddings_aa=encoded_embeddings_aa_rf2, trimmed_attention_mask=trimmed_attention_mask_rf2) #output: [codon_seq_len, C]

        #Concatenate logits from all reading frames along the feature (class logit) dimension
        combined_codon_and_aa_embeddings = torch.cat([logits_rf0, logits_rf1, logits_rf2], dim=-1) #output: [codon_seq_len, 3*C]

        #Map combined frame representations to encoded, shared label space
        logits_encoded_labels = self.linear_transform(combined_codon_and_aa_embeddings) #output: [codon_seq_len, num_encoded_labels]

        #Apply CRF for structured decoding or training (same attention mask applies to all RFs)
        output = self.CRF(
            logits=logits_encoded_labels,
            attention_mask=trimmed_attention_mask_rf0, #Input any trimmed attention mask; applies to same positions as before
            labels=labels)

        return output

In [106]:
def load_and_process_data(test_sample,
                          data_dir,
                          batch_size,
                          max_len = max_len,
                          num_workers_cpu = num_workers_cpu,
                          pin_memory = pin_memory):
    """
    Main function that loads and processes all data efficiently.

    Args:
        test_sample (str): Identifier for the test sample to load.
        data_dir (str): Identifier for error data type.
        max_len (int): Maximum sequence length for tokenization.
        num_workers_cpu (int): Number of CPU workers for data loading.
        pin_memory (bool): Whether to pin memory for DataLoader.
    """
    # Load data
    test_set = pd.read_csv(f"../../../data/processed_data/reads_processed/test/{data_dir}/csv/{test_sample}.csv.gz",
                       index_col=None,
                       compression="gzip")
    
    print("Data samples: ", test_set.shape[0])
    
    # Create tokenizer once and reuse
    tokenizer = AutoTokenizer.from_pretrained(
        "facebook/esm2_t6_8M_UR50D",
        do_lower_case=False,
    )
    
    # Process training data
    test_data = encode_data(test_set, max_len, tokenizer)
    
    # Clear intermediate data to save memory
    del test_set

    test_loader = DataLoader(
        test_data, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers_cpu,  
        pin_memory=pin_memory)
    
    return test_loader

In [107]:
def load_model(model_name_ckpt, error_type, device=device, esm2_model = esm2_model_name):

    """ 
    Load in a trained model for inference.

    Args:
        model_type (str): "with_codon_input" or "without_codon_input"
        model_name_ckpt (str): name of the model checkpoint file
        error_model (str): "model_with_errors" or "model_without_errors"

    Returns: 
        model (nn.Module): The loaded model ready for inference.
    """

    if error_type == "model_with_errors": #implement
        num_labels = 6
    else:
        num_labels = 4
    
    with open('../../../data/processed_data/model_data/shared_crf/model_with_errors/label_mappings/mapping_to_3d_vector.pkl', "rb") as mapping_file: #implemment for with/without errors
        mapping_dict_to_class = pickle.load(mapping_file)
    
    num_encoded_labels = len(mapping_dict_to_class.keys())
    print(f"Number of encoded label classes: {num_encoded_labels}")

    #Set optimized hyperparameters
    num_layers = 6
    n_attention_heads = 4
    dropout_rate_1 = 0.3
    dropout_rate_2 = 0.3
    act_function = "gelu"
        
    model = CDSPredictor(esm2_model=esm2_model,
                         num_layers = num_layers,
                         n_attention_heads = n_attention_heads,
                         dropout_rate_1 = dropout_rate_1, 
                         dropout_rate_2 = dropout_rate_2,
                         act_function = act_function,
                         num_encoded_labels = num_encoded_labels,
                         encoded_labels_mapping = mapping_dict_to_class)
        
    
    model.to(device)
    #Load in parameters from trained model
    model.load_state_dict(torch.load(f"../../../data/processed_data/model_data/shared_crf/{error_type}/models/{model_name_ckpt}", map_location=device), strict=False) #MODIFY

    return model, mapping_dict_to_class

In [108]:
def get_actual_sequence_length(input_ids, eos_token_id=2):
    """Find actual sequence length by locating EOS token"""
    actual_lengths = []
    for seq in input_ids:
        # Find EOS token position
        eos_positions = (seq == eos_token_id).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            # Actual length is EOS position - 1 (excluding CLS)
            actual_length = eos_positions[0].item() - 1
        else:
            # If no EOS found, use full sequence minus CLS
            actual_length = len(seq) - 1
        actual_lengths.append(max(1, actual_length))  # Ensure at least 1
    return actual_lengths

# for inference loop:
def trim_predictions_by_eos(predictions, input_ids):
    actual_lengths = get_actual_sequence_length(input_ids, eos_token_id=2)
    trimmed_predictions = []
    for pred_seq, length in zip(predictions, actual_lengths):
        trimmed_pred = pred_seq[:length]
        trimmed_predictions.append(trimmed_pred)
    return trimmed_predictions

In [109]:
def get_predicted_cds_coords(labels_rf0, labels_rf1, labels_rf2):
    """ 
    Updated function to get predicted CDS coordinates from model labels.
    """
    coords = []
    transition_positions = dict()
    transition_positions['start_codon'] = []
    transition_positions['stop_codon'] = []
    transition_positions['indel_start'] = []
    transition_positions['indel_stop'] = []


    for rf, labels in enumerate([labels_rf0, labels_rf1, labels_rf2]):
        labels = np.array(labels)
        in_cds = False
        start = None
        
        for i, label in enumerate(labels):
            if label in [1, 2, 4]:  # Start codon (2) or inside CDS (1) or indel-caused start (4)
                if not in_cds:
                    in_cds = True
                    start = i
                    if label == 2:
                        transition_positions['start_codon'].append(i*3 + rf + 1)
                    elif label == 4:
                        transition_positions['indel_start'].append(i*3 + rf + 1)

            elif label in [3, 5]:  # Stop codon (3) or indel-caused stop (5)
                if in_cds:
                    # End the current CDS (inclusive of stop codon)
                    end = i + 1  # +1 to include the stop codon position
                    coords.append([start*3 + rf + 1, end * 3 + rf, str(rf)])
                    in_cds = False
                    start = None

                if label == 3:
                    transition_positions['stop_codon'].append(i*3 + 2 + rf + 1) #+2 due to end of codon
                elif label == 5:
                    transition_positions['indel_stop'].append(i*3 + 2 + rf + 1) #+2 due to end of codon

            else:  # label not in [1, 2, 3, 4, 5] - outside CDS
                if in_cds:
                    # End CDS without stop codon
                    end = i
                    coords.append([start*3 + rf + 1, end * 3 + rf, str(rf)])
                    in_cds = False
                    start = None
        
        # Handle case where CDS extends to end of sequence without stop codon
        if in_cds:
            coords.append([(start*3) + rf + 1, len(labels) * 3 + rf, str(rf)])
    
    sorted_coords = sorted(coords, key=lambda x: x[0])
    transition_positions = {k: sorted(v) for k, v in transition_positions.items()}
    
    return sorted_coords, transition_positions

In [110]:
@dataclass
class CDSSegment:
    start: int
    end: int
    frame: int
    start_type: str  # 'start_codon', 'indel_start', 'internal'
    end_type: str    # 'stop_codon', 'indel_stop', 'internal'
    group_id: Optional[str] = None  # Links related fragments
    indel_type: Optional[str] = None  # 'insertion', 'deletion', None

@dataclass
class Transition:
    type: str
    start_position: int
    end_position: int
    frame: int

@dataclass
class UncertainRegion:
    start: int
    end: int
    overlapping_frames: List[int]
    reason: str

def get_cds_coords(labels_rf0, labels_rf1, labels_rf2):
    """
    Enhanced function to get predicted CDS coordinates with frameshift handling and uncertainty detection.
    
    Parameters:
    - labels_rf0, labels_rf1, labels_rf2: prediction arrays for each reading frame
    - uncertainty_threshold: minimum overlap length to flag as uncertain (in nucleotides)
    
    Returns:
    - segments: List of CDSSegment objects
    - uncertain_regions: List of UncertainRegion objects
    - transition_positions: Dict of transition types and positions
    """
    
    #Initialize
    uncertain_regions = []
    transition_positions = {
        'start_codon': [],
        'stop_codon': [],
        'indel_start': [],
        'indel_stop': []}
    all_cds_fragments = []
    transitions_info = []
    
    #Get CDS fragment coordinates for each frame
    for rf, labels in enumerate([labels_rf0, labels_rf1, labels_rf2]):
        labels = np.array(labels)
        frame_segments, start_stop_codon_transitions = _extract_segments_from_frame(labels, rf, transition_positions)
        all_cds_fragments.extend(frame_segments)
        transitions_info.extend(start_stop_codon_transitions)
    
    #Sort CDS fragments by start position
    all_cds_fragments.sort(key=lambda x: x.start)
    
    #Connect fragments interrupted by frameshifts
    connected_segments = _connect_frameshift_segments(all_cds_fragments)
    
    # Create uncertain regions and trim segments for connected groups
    uncertain_regions, transitions_info = _create_uncertain_regions_from_groups(connected_segments, transitions_info)
    
    # Sort final segments
    connected_segments.sort(key=lambda x: x.start)

    #Sort start and stop codon positions
    transitions_info.sort(key=lambda x: x.start_position)
    
    return connected_segments, uncertain_regions, transitions_info, transition_positions

def _extract_segments_from_frame(labels, rf, transition_positions):
    """Extract CDS segments from a single reading frame."""
    segments = []
    start_stop_codon_transitions = []
    in_cds = False
    start = None
    start_type = None
    
    for i, label in enumerate(labels):
        nt_pos = i * 3 + rf + 1  # Convert to nucleotide position (1-indexed)
        
        if label in [1, 2, 4]:  # Start of CDS or inside CDS
            if not in_cds:
                in_cds = True
                start = nt_pos
                if label == 2:
                    start_type = 'start_codon'
                    transition_positions['start_codon'].append(nt_pos)
                    start_stop_codon_transition = Transition(
                        type = "start_codon", 
                        start_position = nt_pos, 
                        end_position = nt_pos + 2,
                        frame = rf)
                    start_stop_codon_transitions.append(start_stop_codon_transition)

                elif label == 4:
                    start_type = 'indel_start'
                    transition_positions['indel_start'].append(nt_pos)
                else:  # label == 1, coding but no explicit start
                    start_type = 'internal'
        
        elif label in [3, 5, 0]:  # End of CDS or non-coding
            if in_cds:
                if label == 3:
                    end_type = 'stop_codon'
                    end = nt_pos + 2  # Include stop codon
                    transition_positions['stop_codon'].append(end)
                    start_stop_codon_transition = Transition(
                        type = "stop_codon", 
                        start_position = nt_pos, 
                        end_position = end,
                        frame = rf)
                    start_stop_codon_transitions.append(start_stop_codon_transition)

                elif label == 5:
                    end_type = 'indel_stop'
                    end = nt_pos + 2
                    transition_positions['indel_stop'].append(end)
                else:  # label == 0, transition to non-coding
                    end_type = 'internal'
                    end = nt_pos - 1
                
                segment = CDSSegment(
                    start=start,
                    end=end,
                    frame=rf,
                    start_type=start_type,
                    end_type=end_type
                )
                segments.append(segment)
                
                in_cds = False
                start = None
                start_type = None
    
    # Handle case where CDS extends to end of sequence
    if in_cds:
        end = len(labels) * 3 + rf
        segment = CDSSegment(
            start=start,
            end=end,
            frame=rf,
            start_type=start_type,
            end_type='internal'
        )
        segments.append(segment)
    
    return segments, start_stop_codon_transitions

def _create_uncertain_regions_from_groups(segments, transitions):
    """Create uncertain regions between connected frameshift segments and trim overlapping parts."""
    uncertain_regions = []
    
    # Group segments by group_id
    groups = defaultdict(list)
    for segment in segments:
        if segment.group_id:
            groups[segment.group_id].append(segment)
    
    # Process each group to create uncertain regions and trim segments
    for group_id, group_segments in groups.items():
        if len(group_segments) < 2:
            continue
            
        # Sort segments in the group by start position
        group_segments.sort(key=lambda x: x.start)
        
        # Process consecutive pairs of segments in the group
        for i in range(len(group_segments) - 1):
            seg1 = group_segments[i]
            seg2 = group_segments[i + 1]
            
            # Determine the uncertain region between these two segments
            if seg1.end >= seg2.start:
                # Overlapping case - need to trim at codon boundaries
                
                # Find the last complete codon in seg1's frame before overlap
                overlap_start = seg2.start
                # Calculate the last codon boundary in seg1's frame before overlap
                positions_before_overlap = overlap_start - seg1.start
                complete_codons_in_seg1 = positions_before_overlap // 3
                seg1_trim_end = seg1.start + (complete_codons_in_seg1 * 3) - 1
                
                # Find the first complete codon in seg2's frame after overlap  
                overlap_end = seg1.end
                # Calculate positions from seg2 start to end of overlap
                positions_in_overlap = overlap_end - seg2.start + 1
                # Round up to next codon boundary
                codons_to_skip = (positions_in_overlap + 2) // 3  # +2 for ceiling division
                seg2_trim_start = seg2.start + (codons_to_skip * 3)
                
                # Only trim if we have valid boundaries
                if seg1_trim_end >= seg1.start and seg2_trim_start <= seg2.end:
                    # Update segment boundaries
                    original_seg1_end = seg1.end
                    original_seg2_start = seg2.start
                    
                    seg1.end = seg1_trim_end
                    seg2.start = seg2_trim_start
                    
                    # Create uncertain region for the removed area
                    uncertain_start = seg1.end + 1
                    uncertain_end = seg2.start - 1
                    
                    if uncertain_end > uncertain_start:
                        uncertain_region = UncertainRegion(
                            start=uncertain_start,
                            end=uncertain_end,
                            overlapping_frames=[seg1.frame, seg2.frame],
                            reason=f"Frameshift overlap between RF{seg1.frame} and RF{seg2.frame}"
                        )
                        uncertain_regions.append(uncertain_region)

                    elif uncertain_end == uncertain_start:
                        transition = Transition(
                            type = "insertion", 
                            start_position = uncertain_start, 
                            end_position = uncertain_end,
                            frame = seg1.frame)
                        transitions.append(transition)
                
            else:
                # Non-overlapping case - gap between segments
                gap_start = seg1.end + 1
                gap_end = seg2.start - 1
                
                if gap_end > gap_start:  # Only create if there's actually a gap
                    uncertain_region = UncertainRegion(
                        start=gap_start,
                        end=gap_end,
                        overlapping_frames=[seg1.frame, seg2.frame],
                        reason=f"Frameshift gap between RF{seg1.frame} and RF{seg2.frame}"
                    )
                    uncertain_regions.append(uncertain_region)

                elif gap_end == gap_start:
                    transition = Transition(
                        type = "insertion", 
                        start_position = gap_start, 
                        end_position = gap_end,
                        frame = seg1.frame)
                    transitions.append(transition)
                
    return uncertain_regions, transitions

def detect_indel_type(from_frame, to_frame):
    """
    Detect indel type based on reading frame transition.
    
    Insertions cause forward jumps: 0->1, 1->2, 2->0
    Deletions cause backward jumps: 0->2, 1->0, 2->1
    """
    if from_frame == to_frame:
        return None
    
    # Forward jumps (insertion)
    forward_jumps = {(0, 1), (1, 2), (2, 0)}
    # Backward jumps (deletion)  
    backward_jumps = {(0, 2), (1, 0), (2, 1)}
    
    transition = (from_frame, to_frame)
    
    if transition in forward_jumps:
        return 'insertion'
    elif transition in backward_jumps:
        return 'deletion'
    else:
        return 'complex'  # Multiple frame shifts

def _connect_frameshift_segments(segments):
    """Attempt to connect segments that might be part of the same CDS interrupted by frameshifts."""
    connected_segments = []
    used_segments = set()
    group_counter = 1
    
    for i, segment in enumerate(segments):
        if i in used_segments:
            continue
            
        current_group = []
        current_group.append(segment)
        used_segments.add(i)
        
        # Look for segments that could be connected
        if segment.end_type == 'indel_stop':
            # Look for nearby indel_start segments in different frames
            for j, other_segment in enumerate(segments[i+1:], i+1):
                if (j not in used_segments and 
                    other_segment.start_type == 'indel_start' and
                    other_segment.frame != segment.frame and
                    abs(other_segment.start - segment.end) <= 30):  # Allow reasonable gap between predicted fragmented segments
                    
                    # Detect indel type based on direction of jump in frame
                    indel_type = detect_indel_type(segment.frame, other_segment.frame)
                    
                    # Update both segments with indel type (keep original start/end types)
                    segment.indel_type = indel_type
                    other_segment.indel_type = indel_type
                    
                    current_group.append(other_segment)
                    used_segments.add(j)
                    
                    # Continue looking for more connected segments from this one
                    last_segment = other_segment
                    for k, next_segment in enumerate(segments[j+1:], j+1):
                        if (k not in used_segments and
                            last_segment.end_type == 'indel_stop' and
                            next_segment.start_type == 'indel_start' and
                            next_segment.frame != last_segment.frame and
                            abs(next_segment.start - last_segment.end) <= 30):
                            
                            next_indel_type = detect_indel_type(last_segment.frame, next_segment.frame)
                            last_segment.indel_type = next_indel_type
                            next_segment.indel_type = next_indel_type
                            
                            current_group.append(next_segment)
                            used_segments.add(k)
                            last_segment = next_segment
                        else:
                            break
                    break
        
        # Assign group ID if multiple segments are connected
        if len(current_group) > 1:
            group_id = f"group_{group_counter}"
            for seg in current_group:
                seg.group_id = group_id
            group_counter += 1
        
        connected_segments.extend(current_group)
    
    return connected_segments

def write_enhanced_gff(segments, uncertain_regions, transitions_info, read_name, cds_coords, seq_errors, outfile_gff):
    """Write segments and uncertain regions to GFF file with enhanced annotations."""
    
    counter_cds_frags_interrupted = dict()

    # Write CDS segments
    for i, segment in enumerate(segments):
        # Create attributes
        attributes = []
        #attributes.append(f"ID=cds_{read_name}_{i}")
        #attributes.append(f"product=predicted_protein_fragment")
        attributes.append(f"start_type={segment.start_type}")
        attributes.append(f"end_type={segment.end_type}")
        
        # Add group ID if this segment is part of a connected group
        if segment.group_id:
            if segment.group_id not in counter_cds_frags_interrupted.keys():
                counter_cds_frags_interrupted[segment.group_id] = 0
            
            else:
                counter_cds_frags_interrupted[segment.group_id] += 1
            attributes.append(f"group_id={segment.group_id}.{counter_cds_frags_interrupted[segment.group_id]}")

        # Add indel type if detected
        if segment.indel_type:
            attributes.append(f"indel_type={segment.indel_type}")
            #attributes.append(f"Note=Frameshift Caused by Indel")
        
        attributes.append(f"ref={cds_coords}")
        attributes.append(f"seq_errors={seq_errors}")
        
        attr_string = ";".join(attributes)
        
        outfile_gff.write(
            f"{read_name}\tFragmentPredictor\tCDS\t{segment.start}\t{segment.end}\t"
            f".\t+\t{segment.frame}\t{attr_string}\n"
        )
    
    # Write start, stop and insertion positions as separate features
    for i, transition in enumerate(transitions_info):
        attributes = []
        attributes.append(f"ID={transition.type}_{read_name}_{i}") 

        attr_string = ";".join(attributes)

        outfile_gff.write(
            f"{read_name}\tFragmentPredictor\t{transition.type}\t{transition.start_position}\t{transition.end_position}\t"
            f".\t+\t.\t{attr_string}\n"
        )

    # Write uncertain regions as separate features
    for i, region in enumerate(uncertain_regions):
        attributes = []
        #attributes.append(f"ID=uncertain_{read_name}_{i}")
        attributes.append(f"Note=Uncertain region: {region.reason}")
        attributes.append(f"overlapping_frames={','.join(map(str, region.overlapping_frames))}")
        
        # Add group information if the uncertain region involves grouped segments
        involved_groups = set()
        for segment in segments:
            if (segment.group_id and 
                not (segment.end < region.start or segment.start > region.end)):
                involved_groups.add(segment.group_id)
        
        if involved_groups:
            attributes.append(f"involved_groups={','.join(involved_groups)}")
        
        attr_string = ";".join(attributes)
        
        outfile_gff.write(
            f"{read_name}\tFragmentPredictor\tuncertain_region\t{region.start}\t{region.end}\t"
            f".\t+\t.\t{attr_string}\n"
        )

# Example usage function to replace your existing processing loop
def process_predictions_enhanced(predictions_rf0, predictions_rf1, predictions_rf2, 
                               read_names, cds_coords, seq_errors, outfile_gff, batch_size):
    """
    Processing function...
    """
    
    for i in range(min(batch_size, len(cds_coords))):
        # Get enhanced predictions

        segments, uncertain_regions, transitions_info, transition_positions = get_cds_coords(
            predictions_rf0[i], predictions_rf1[i], predictions_rf2[i])
        
        # Write to GFF
        write_enhanced_gff(segments, uncertain_regions, transitions_info, read_names[i], 
                        cds_coords[i], seq_errors[i], outfile_gff)


In [111]:
def run_model_predictions(data_dir, model, mapping_dict_to_class, test_samples = test_samples, batch_size=256):
    for test_sample in test_samples:
        print(test_sample)
        test_loader = load_and_process_data(test_sample,
                                            data_dir = data_dir,
                                            batch_size = batch_size,
                                            max_len = max_len,
                                            num_workers_cpu = num_workers_cpu,
                                            pin_memory = pin_memory)

        dir_path = f"../../../data/processed_data/predictions/raw_predictions/model_shared_crf_preds/{data_dir}/{model_name_ckpt[:-4]}/"
        os.makedirs(dir_path, exist_ok=True)
        outfile_gff = open(f"{dir_path}/predictions_{test_sample}.gff", "w")
        outfile_gff.write("##gff-version 3\n")

        with torch.no_grad():
            model.eval()
            for counter, batch in tqdm(enumerate(test_loader), total=len(test_loader)):

                aa_encoding_rf0 = batch['aa_encodings_rf0']['input_ids'].to(device)
                rf0_attention_mask = batch['aa_encodings_rf0']['attention_mask'].to(device)
                aa_encoding_rf1 = batch['aa_encodings_rf1']['input_ids'].to(device)
                rf1_attention_mask = batch['aa_encodings_rf1']['attention_mask'].to(device)
                aa_encoding_rf2 = batch['aa_encodings_rf2']['input_ids'].to(device)
                rf2_attention_mask = batch['aa_encodings_rf2']['attention_mask'].to(device)

                
                cds_coords = batch['cds_coords']
                read_names = batch['read_name']
                seq_errors = batch['seq_errors']
                
                nt_encoding_rf0 = batch['nt_encodings_rf0'].to(device)
                nt_encoding_rf1 = batch['nt_encodings_rf1'].to(device)
                nt_encoding_rf2 = batch['nt_encodings_rf2'].to(device)

                #Predict with shared crf
                outputs = model(nt_encoding_rf0, aa_encoding_rf0, rf0_attention_mask, 
                                  nt_encoding_rf1, aa_encoding_rf1, rf1_attention_mask, 
                                  nt_encoding_rf2, aa_encoding_rf2, rf2_attention_mask)


                    
                predictions_encoded = outputs["predictions"]

                # Decode predictions and split into rf0/rf1/rf2
                preds_rf0, preds_rf1, preds_rf2 = [], [], []

                for preds_sample in predictions_encoded:
                    preds = [mapping_dict_to_class[p] for p in preds_sample]  # decode sequence
                    preds_rf0.append([rf[0] for rf in preds])
                    preds_rf1.append([rf[1] for rf in preds])
                    preds_rf2.append([rf[2] for rf in preds])

                preds_rf0 = trim_predictions_by_eos(preds_rf0, aa_encoding_rf0)
                preds_rf1 = trim_predictions_by_eos(preds_rf1, aa_encoding_rf1)
                preds_rf2 = trim_predictions_by_eos(preds_rf2, aa_encoding_rf2)

                #print(read_names)
                #print(preds_rf0)
                #print(preds_rf1)
                #print(preds_rf2)
                #print()

                process_predictions_enhanced(
                    preds_rf0, preds_rf1, preds_rf2,
                    read_names, cds_coords, seq_errors, outfile_gff, batch_size
                )

        outfile_gff.close()

In [112]:
model, mapping_dict_to_class = load_model(model_name_ckpt, "model_with_errors")

for data_dir in ["with_errors_5e-06i_0.004s_300bp",
                 "with_errors_1.25e-05i_0.01s_300bp",
                 "with_errors_3.75e-05i_0.03s_300bp",
                 "without_errors_300bp"]:
    
    print(data_dir, flush = True)
    run_model_predictions(data_dir, model, mapping_dict_to_class)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of encoded label classes: 69
Legal transitions: 627
Illegal transitions: 4134
Total transitions: 4761
Percentage legal: 13.2%
Frequent self-transitions: 4
Frequent transition labels: [0, 1, 6, 26]
with_errors_5e-06i_0.004s_300bp


/Users/nrt204/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


GCF_001277215.2
Data samples:  26835


/Users/nrt204/miniconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28

...

/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
100%|██████████| 24/24 [01:45<00:00,  4.39s/it]

with_errors_1.25e-05i_0.01s_300bp


GCF_001277215.2
Data samples:  26787


/Users/nrt204/miniconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(encoding)
/var/folders/2g/7qq8263d1bd2j1p8_1gc7w3h0000gn/T/ipykernel_32066/1399213979.py:28

KeyboardInterrupt: 